# Processing of a resting-state EEG dataset for sleep deprivation

Bad channels are interpolated

## PRE PROCESSING

#### The intention is to provide epochs(4 seconds) to the machine and try predicting if the case is sleep deprivation or normal sleep.

In [1]:
from matplotlib import pyplot as plt
import os
import pandas as pd
import numpy as np
from scipy import signal
import mne
import mne_icalabel
from mne.preprocessing import ICA
from autoreject import Ransac  
from mne_icalabel import label_components
%matplotlib inline

In [2]:
# lists to collect data
X_frontal_SD_list = []
X_temporal_SD_list = []
X_parietal_SD_list = []
X_occipital_SD_list = []
X_central_SD_list = []

X_frontal_NS_list = []
X_temporal_NS_list = []
X_parietal_NS_list = []
X_occipital_NS_list = []
X_central_NS_list = []

In [3]:
n_epochs = 0
for i in range(1, 72):
    if i == 28 or i== 1 or i== 44 or i==39 or i==43 or i in [7,8,11,13,18,24,29,31,35,36,37,38,40,47,48,50,54,58,59,63,67,70,71]: #46
        continue  # Skip participants due to missing data
    for j in range(1, 3):
        
        #Define file path
        file_path = f'D:\\BCI\\TASKS\\Task-8\\ds004902_data\\sub-{i:02d}\\ses-{j}\\eeg\\sub-{i:02d}_ses-{j}_task-eyesopen_eeg.set'#The d indicates that the value is a decimal integer, while the 02 specifies that the output should be padded with leading zeros if necessary to reach a width of two characters.
        epochs_file = f'D:\\BCI\\TASKS\\Task-8\\processed_epochs_60\\sub-{i:02d}_ses-{j}_epochs.fif'  # Path to save/load processed epochs

        
        eeg_channels = ["Fp1", "AF3", "AF7", "Fz", "F1", "F3", "F5", "F7","FC1", "FC3", "FC5", "FT7","Cz", "C1", "C3", "C5", "T7","CP1", "CP3", "CP5", "TP7", "TP9","Pz", "P1", "P3", "P5", "P7","PO3", "PO7", "Oz", "O1","Fpz", "Fp2", "AF4", "AF8","F2", "F4", "F6", "F8","FC2", "FC4", "FC6", "FT8","C2", "C4", "C6", "T8","CPz", "CP2", "CP4", "CP6","TP8", "TP10","P2", "P4", "P6", "P8","POz", "PO4", "PO8", "O2"]
               
        if os.path.exists(epochs_file):
            print(f"Loading processed epochs from {epochs_file}")
            epochs = mne.read_epochs(epochs_file, preload=True)
        else:
            print(f"\nProcessing: {file_path}\n")

            #Define EEG channels
            eeg_channels = ["Fp1", "AF3", "AF7", "Fz", "F1", "F3", "F5", "F7","FC1", "FC3", "FC5", "FT7","Cz", "C1", "C3", "C5", "T7","CP1", "CP3", "CP5", "TP7", "TP9","Pz", "P1", "P3", "P5", "P7","PO3", "PO7", "Oz", "O1","Fpz", "Fp2", "AF4", "AF8","F2", "F4", "F6", "F8","FC2", "FC4", "FC6", "FT8","C2", "C4", "C6", "T8","CPz", "CP2", "CP4", "CP6","TP8", "TP10","P2", "P4", "P6", "P8","POz", "PO4", "PO8", "O2"]
            
            #Load EEG data
            raw = mne.io.read_raw_eeglab(file_path, preload=True)
            eeg_raw=raw.pick(mne.pick_channels(raw.info['ch_names'], include=eeg_channels))
            
            #Montage setting
            montage = mne.channels.make_standard_montage('standard_1020')
            eeg_raw.set_montage(montage)

            #Filtering
            eeg_raw.filter(1., 40.)#, fir_design='firwin')

            #Average Referencing
            eeg_raw.set_eeg_reference('average', projection=True)
            eeg_raw.apply_proj()
            
            #EPOCHING
            epochs = mne.make_fixed_length_epochs(eeg_raw, duration=4.0, overlap=0.0,preload=True)
            
            #RANSAC for bad channel detection
            ransac = Ransac(verbose=False, n_jobs=1)
            ransac.fit(epochs)
            raw.info['bads'].extend(ransac.bad_chs_)   

            #interpolate bad channels
            raw.interpolate_bads(reset_bads=True)

            
            #ICA for artifact removal
            ica = mne.preprocessing.ICA(n_components=20, method='infomax', fit_params=dict(extended=True), random_state=42, max_iter=1000)
            ica.fit(epochs)

            ic_labels = label_components(epochs, ica, method='iclabel')
            exclude_idx = []
            for idx, label in enumerate(ic_labels["labels"]):
                if label not in ["brain","other","muscle artifact"] and ic_labels["y_pred_proba"][idx] > 0.5:
                    exclude_idx.append(idx)

            ica.apply(epochs,exclude= exclude_idx)        

            # Save processed epochs
            os.makedirs(os.path.dirname(epochs_file), exist_ok=True) #creates the necessary directories for the file path if they don't exist.
            epochs.save(epochs_file, overwrite=True)
            print(f"Saved processed epochs to {epochs_file}")

        if j==1:
            labels = 0
        elif j==2:
            labels = 1
        labels = pd.DataFrame({'label': [labels]*len(epochs)})

        # Extract data and labels
        data = epochs.get_data()  # shape: (n_epochs, n_channels, n_times)

        # 1. Frontal: 
        frontal_eeg_channels = [ch for ch in eeg_channels if ch.startswith(('Fp', 'AF', 'F', 'FC')) and not ch.startswith('FT')] 
        data_frontal = epochs.copy().pick(frontal_eeg_channels)

        # 2. Temporal: 
        temporal_eeg_channels = [ch for ch in eeg_channels if ch.startswith(('T', 'FT', 'TP'))]
        data_temporal = epochs.copy().pick(temporal_eeg_channels)

        # 3. Parietal: 
        parietal_eeg_channels = [ch for ch in eeg_channels if ch.startswith(('P', 'CP', 'PO'))]
        data_parietal = epochs.copy().pick(parietal_eeg_channels)

        # 4. Occipital:
        occipital_eeg_channels = [ch for ch in eeg_channels if ch.startswith('O')]
        data_occipital = epochs.copy().pick(occipital_eeg_channels)

        # 5. Central channels:
        central_channels = [ch for ch in eeg_channels if ch.startswith('C') and not ch.startswith('CP')]
        data_central = epochs.copy().pick(central_channels)

        #if (i == 39 or i == 43) and j == 2:
        #    data = signal.decimate(data, 10, axis=2) # Downsample by a factor of 10 to match time points.

        epoch_labels = labels['label'].values  
        
        if i <= 71 and j==2: 
            X_frontal_SD_list.append(data_frontal.get_data())
            X_temporal_SD_list.append(data_temporal.get_data())
            X_parietal_SD_list.append(data_parietal.get_data())
            X_occipital_SD_list.append(data_occipital.get_data())
            X_central_SD_list.append(data_central.get_data())
        
        if i<=71 and j==1:
            X_frontal_NS_list.append(data_frontal.get_data())
            X_temporal_NS_list.append(data_temporal.get_data())
            X_parietal_NS_list.append(data_parietal.get_data())
            X_occipital_NS_list.append(data_occipital.get_data())
            X_central_NS_list.append(data_central.get_data())

# After the loop, concatenate into arrays
X_frontal_SD = np.concatenate(X_frontal_SD_list, axis=0)
X_temporal_SD = np.concatenate(X_temporal_SD_list, axis=0)
X_parietal_SD = np.concatenate(X_parietal_SD_list, axis=0)
X_occipital_SD = np.concatenate(X_occipital_SD_list, axis=0)
X_central_SD = np.concatenate(X_central_SD_list, axis=0)

X_frontal_NS = np.concatenate(X_frontal_NS_list, axis=0)
X_temporal_NS = np.concatenate(X_temporal_NS_list, axis=0)
X_parietal_NS = np.concatenate(X_parietal_NS_list, axis=0)
X_occipital_NS = np.concatenate(X_occipital_NS_list, axis=0)
X_central_NS = np.concatenate(X_central_NS_list, axis=0)
print("Training and testing data processing complete.")

Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-1_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-1_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available
Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated


C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1121323011.py:16: RuntimeWarning: This filename (D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-1_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(epochs_file, preload=True)


Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-2_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-2_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available
Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated


C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1121323011.py:16: RuntimeWarning: This filename (D:\BCI\TASKS\Task-8\processed_epochs_60\sub-02_ses-2_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(epochs_file, preload=True)


Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-03_ses-1_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-03_ses-1_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available
Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated


...

Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated
Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-1_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-1_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available


C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1121323011.py:16: RuntimeWarning: This filename (D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-1_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(epochs_file, preload=True)


Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated
Loading processed epochs from D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-2_epochs.fif
Reading D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-2_epochs.fif ...
    Found the data of interest:
        t =       0.00 ...   59998.00 ms
        0 CTF compensation matrices available


C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1121323011.py:16: RuntimeWarning: This filename (D:\BCI\TASKS\Task-8\processed_epochs_60\sub-69_ses-2_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(epochs_file, preload=True)


Not setting metadata
5 matching events found
No baseline correction applied
0 projection items activated
Training and testing data processing complete.


In [4]:
X_frontal_NS.shape, X_temporal_NS.shape, X_parietal_NS.shape, X_occipital_NS.shape, X_central_NS.shape

((214, 22, 30000),
 (214, 8, 30000),
 (214, 21, 30000),
 (214, 3, 30000),
 (214, 7, 30000))

## FEATURE EXTRACTION 

In [5]:
from scipy import stats

In [6]:
#Time-domain feature extraction functions
def mean(sample_data):
    return np.mean(sample_data, axis=-1)

def std(sample_data):
    return np.std(sample_data, axis=-1)

def zscore(sample_data):
    return stats.zscore(sample_data, axis=-1)

def ptp(sample_data):
    return np.ptp(sample_data, axis=-1)

def min(sample_data):
    return np.min(sample_data, axis=-1)

def max(sample_data):
    return np.max(sample_data, axis=-1)

def var(sample_data):
    return np.var(sample_data, axis=-1)

def rms(sample_data):
    return np.sqrt(np.mean(sample_data**2, axis=-1))

def skewness(sample_data):
    return stats.skew(sample_data, axis=-1)   

def kurtosis(sample_data):
    return stats.kurtosis(sample_data, axis=-1)


def mean_square(sample_data):
    return np.mean(sample_data**2, axis=-1)

def hjorth_params(sample_data):
    first_deriv = np.diff(sample_data, axis=-1)
    second_deriv = np.diff(first_deriv, axis=-1)

    var_zero = var(sample_data)
    var_d1 = var(first_deriv)
    var_d2 = var(second_deriv)
    
    activity = var_zero
    mobility = np.sqrt(var_d1 / var_zero)
    complexity = np.sqrt(var_d2 / var_d1) / mobility
    
    return activity, mobility, complexity

In [7]:
# Frequency-domain feature extraction functions
def bandpower_delta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    delta_band = (0.5, 4)
    mask = (freqs >= delta_band[0]) & (freqs <= delta_band[1])
    delta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return delta_power

def bandpower_theta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    theta_band = (4, 8)
    mask = (freqs >= theta_band[0]) & (freqs <= theta_band[1])
    theta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return theta_power

def bandpower_alpha(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    alpha_band = (8, 13)
    mask = (freqs >= alpha_band[0]) & (freqs <= alpha_band[1])
    alpha_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return alpha_power  

def bandpower_beta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    beta_band = (13, 30)
    mask = (freqs >= beta_band[0]) & (freqs <= beta_band[1])
    beta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return beta_power       

def bandpower_gamma(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    gamma_band = (30, 40)
    mask = (freqs >= gamma_band[0]) & (freqs <= gamma_band[1])
    gamma_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    return gamma_power

def spectral_entropy(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    psd_norm = psd_values / np.sum(psd_values, axis=axis, keepdims=True)
    spectral_entropy = -np.sum(psd_norm * np.log2(psd_norm + 1e-10), axis=axis)
    return spectral_entropy

In [8]:
# Average Bandpower extraction functions
def average_bandpower_delta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    delta_band = (0.5, 4)
    mask = (freqs >= delta_band[0]) & (freqs <= delta_band[1])
    delta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis) # dimensions: (n_epochs, n_channels)
    average_delta_bandpower = np.mean(delta_power, axis= -1)  # Average across channels
    return average_delta_bandpower

def average_bandpower_theta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    theta_band = (4, 8)
    mask = (freqs >= theta_band[0]) & (freqs <= theta_band[1])
    theta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    average_theta_bandpower = np.mean(theta_power, axis=-1)  # Average across channels
    return average_theta_bandpower

def average_bandpower_alpha(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    alpha_band = (8, 13)
    mask = (freqs >= alpha_band[0]) & (freqs <= alpha_band[1])
    alpha_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    average_alpha_bandpower = np.mean(alpha_power, axis=-1)  # Average across channels
    return average_alpha_bandpower      

def average_bandpower_beta(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    beta_band = (13, 30)
    mask = (freqs >= beta_band[0]) & (freqs <= beta_band[1])
    beta_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    average_beta_bandpower = np.mean(beta_power, axis=-1)  # Average across channels
    return average_beta_bandpower

def average_bandpower_gamma(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    gamma_band = (30, 40)
    mask = (freqs >= gamma_band[0]) & (freqs <= gamma_band[1])
    gamma_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    average_gamma_bandpower = np.mean(gamma_power, axis=-1)  # Average across channels
    return average_gamma_bandpower

def average_bandpower_total(sample_data, fs=500, axis=-1):
    freqs, psd_values = signal.welch(sample_data, fs=fs, axis=axis)
    total_band = (0.5, 40)
    mask = (freqs >= total_band[0]) & (freqs <= total_band[1])
    total_power = np.trapezoid(psd_values[..., mask], freqs[mask], axis=axis)
    average_total_bandpower = np.mean(total_power, axis=-1)  # Average across channels
    return average_total_bandpower

#### Frontal

In [9]:
from tqdm import tqdm_notebook
X_frontal_NS_features=[]
for datta in tqdm_notebook(X_frontal_NS):
    features = average_bandpower_delta(datta)
    X_frontal_NS_features.append(features)

X_frontal_NS_delta=np.array(X_frontal_NS_features)
X_frontal_NS_delta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\3613515536.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_frontal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [10]:
from tqdm import tqdm_notebook
X_frontal_NS_features=[]
for datta in tqdm_notebook(X_frontal_NS):
    features = average_bandpower_theta(datta)
    X_frontal_NS_features.append(features)

X_frontal_NS_theta=np.array(X_frontal_NS_features)
X_frontal_NS_theta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\3074469466.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_frontal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [11]:
from tqdm import tqdm_notebook
X_frontal_NS_features=[]
for datta in tqdm_notebook(X_frontal_NS):
    features = average_bandpower_alpha(datta)
    X_frontal_NS_features.append(features)

X_frontal_NS_alpha=np.array(X_frontal_NS_features)
X_frontal_NS_alpha.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\3816245252.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_frontal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [12]:
from tqdm import tqdm_notebook
X_frontal_NS_features=[]
for datta in tqdm_notebook(X_frontal_NS):
    features = average_bandpower_beta(datta)
    X_frontal_NS_features.append(features)

X_frontal_NS_beta=np.array(X_frontal_NS_features)
X_frontal_NS_beta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\991141888.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_frontal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [13]:
from tqdm import tqdm_notebook
X_frontal_NS_features=[]
for datta in tqdm_notebook(X_frontal_NS):
    features = average_bandpower_gamma(datta)
    X_frontal_NS_features.append(features)

X_frontal_NS_gamma=np.array(X_frontal_NS_features)
X_frontal_NS_gamma.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2186554699.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_frontal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [14]:
from tqdm import tqdm_notebook
X_occipital_NS_features=[]
for datta in tqdm_notebook(X_occipital_NS):
    features = average_bandpower_delta(datta)
    X_occipital_NS_features.append(features)

X_occipital_NS_delta=np.array(X_occipital_NS_features)
X_occipital_NS_delta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1731542266.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_occipital_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [15]:
from tqdm import tqdm_notebook
X_occipital_NS_features=[]
for datta in tqdm_notebook(X_occipital_NS):
    features = average_bandpower_theta(datta)
    X_occipital_NS_features.append(features)

X_occipital_NS_theta=np.array(X_occipital_NS_features)
X_occipital_NS_theta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1062760812.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_occipital_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [16]:
from tqdm import tqdm_notebook
X_occipital_NS_features=[]
for datta in tqdm_notebook(X_occipital_NS):
    features = average_bandpower_alpha(datta)
    X_occipital_NS_features.append(features)

X_occipital_NS_alpha=np.array(X_occipital_NS_features)
X_occipital_NS_alpha.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2253058283.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_occipital_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [17]:
from tqdm import tqdm_notebook
X_occipital_NS_features=[]
for datta in tqdm_notebook(X_occipital_NS):
    features = average_bandpower_beta(datta)
    X_occipital_NS_features.append(features)

X_occipital_NS_beta=np.array(X_occipital_NS_features)
X_occipital_NS_beta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1596264442.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_occipital_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [18]:
from tqdm import tqdm_notebook
X_occipital_NS_features=[]
for datta in tqdm_notebook(X_occipital_NS):
    features = average_bandpower_gamma(datta)
    X_occipital_NS_features.append(features)

X_occipital_NS_gamma=np.array(X_occipital_NS_features)
X_occipital_NS_gamma.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1558315074.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_occipital_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

#### parietal

In [19]:
from tqdm import tqdm_notebook
X_parietal_NS_features=[]
for datta in tqdm_notebook(X_parietal_NS):
    features = average_bandpower_delta(datta)
    X_parietal_NS_features.append(features)

X_parietal_NS_delta=np.array(X_parietal_NS_features)
X_parietal_NS_delta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\295426021.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_parietal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [20]:
from tqdm import tqdm_notebook
X_parietal_NS_features=[]
for datta in tqdm_notebook(X_parietal_NS):
    features = average_bandpower_theta(datta)
    X_parietal_NS_features.append(features)

X_parietal_NS_theta=np.array(X_parietal_NS_features)
X_parietal_NS_theta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\956604678.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_parietal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [21]:
from tqdm import tqdm_notebook
X_parietal_NS_features=[]
for datta in tqdm_notebook(X_parietal_NS):
    features = average_bandpower_alpha(datta)
    X_parietal_NS_features.append(features)

X_parietal_NS_alpha=np.array(X_parietal_NS_features)
X_parietal_NS_alpha.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2043724890.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_parietal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [22]:
from tqdm import tqdm_notebook
X_parietal_NS_features=[]
for datta in tqdm_notebook(X_parietal_NS):
    features = average_bandpower_beta(datta)
    X_parietal_NS_features.append(features)

X_parietal_NS_beta=np.array(X_parietal_NS_features)
X_parietal_NS_beta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2134154589.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_parietal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [23]:
from tqdm import tqdm_notebook
X_parietal_NS_features=[]
for datta in tqdm_notebook(X_parietal_NS):
    features = average_bandpower_gamma(datta)
    X_parietal_NS_features.append(features)

X_parietal_NS_gamma=np.array(X_parietal_NS_features)
X_parietal_NS_gamma.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2013756118.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_parietal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

#### Temporal

In [24]:
from tqdm import tqdm_notebook
X_temporal_NS_features=[]
for datta in tqdm_notebook(X_temporal_NS):
    features = average_bandpower_delta(datta)
    X_temporal_NS_features.append(features)

X_temporal_NS_delta=np.array(X_temporal_NS_features)
X_temporal_NS_delta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1852679749.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_temporal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [25]:
from tqdm import tqdm_notebook
X_temporal_NS_features=[]
for datta in tqdm_notebook(X_temporal_NS):
    features = average_bandpower_theta(datta)
    X_temporal_NS_features.append(features)

X_temporal_NS_theta=np.array(X_temporal_NS_features)
X_temporal_NS_theta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\4293974839.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_temporal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [26]:
from tqdm import tqdm_notebook
X_temporal_NS_features=[]
for datta in tqdm_notebook(X_temporal_NS):
    features = average_bandpower_alpha(datta)
    X_temporal_NS_features.append(features)

X_temporal_NS_alpha=np.array(X_temporal_NS_features)
X_temporal_NS_alpha.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1530542361.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_temporal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [27]:
from tqdm import tqdm_notebook
X_temporal_NS_features=[]
for datta in tqdm_notebook(X_temporal_NS):
    features = average_bandpower_beta(datta)
    X_temporal_NS_features.append(features)

X_temporal_NS_beta=np.array(X_temporal_NS_features)
X_temporal_NS_beta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1007577858.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_temporal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [28]:
from tqdm import tqdm_notebook
X_temporal_NS_features=[]
for datta in tqdm_notebook(X_temporal_NS):
    features = average_bandpower_gamma(datta)
    X_temporal_NS_features.append(features)

X_temporal_NS_gamma=np.array(X_temporal_NS_features)
X_temporal_NS_gamma.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1495793089.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_temporal_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

#### Central

In [29]:
from tqdm import tqdm_notebook
X_central_NS_features=[]
for datta in tqdm_notebook(X_central_NS):
    features = average_bandpower_delta(datta)
    X_central_NS_features.append(features)

X_central_NS_delta=np.array(X_central_NS_features)
X_central_NS_delta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2091627785.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_central_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [30]:
from tqdm import tqdm_notebook
X_central_NS_features=[]
for datta in tqdm_notebook(X_central_NS):
    features = average_bandpower_theta(datta)
    X_central_NS_features.append(features)

X_central_NS_theta=np.array(X_central_NS_features)
X_central_NS_theta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1455052227.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_central_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [31]:
from tqdm import tqdm_notebook
X_central_NS_features=[]
for datta in tqdm_notebook(X_central_NS):
    features = average_bandpower_alpha(datta)
    X_central_NS_features.append(features)

X_central_NS_alpha=np.array(X_central_NS_features)
X_central_NS_alpha.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1980510834.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_central_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [32]:
from tqdm import tqdm_notebook
X_central_NS_features=[]
for datta in tqdm_notebook(X_central_NS):
    features = average_bandpower_beta(datta)
    X_central_NS_features.append(features)

X_central_NS_beta=np.array(X_central_NS_features)
X_central_NS_beta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2682824565.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_central_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [33]:
from tqdm import tqdm_notebook
X_central_NS_features=[]
for datta in tqdm_notebook(X_central_NS):
    features = average_bandpower_gamma(datta)
    X_central_NS_features.append(features)

X_central_NS_gamma=np.array(X_central_NS_features)
X_central_NS_gamma.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\3582172607.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_central_NS):


  0%|          | 0/214 [00:00<?, ?it/s]

(214,)

In [34]:
from tqdm import tqdm_notebook
X_frontal_SD_features=[]
for datta in tqdm_notebook(X_frontal_SD):
    features = average_bandpower_delta(datta)
    X_frontal_SD_features.append(features)

X_frontal_SD_delta=np.array(X_frontal_SD_features)
X_frontal_SD_delta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2064858051.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_frontal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [35]:
from tqdm import tqdm_notebook
X_frontal_SD_features=[]
for datta in tqdm_notebook(X_frontal_SD):
    features = average_bandpower_theta(datta)
    X_frontal_SD_features.append(features)

X_frontal_SD_theta=np.array(X_frontal_SD_features)
X_frontal_SD_theta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1298708945.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_frontal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [36]:
from tqdm import tqdm_notebook
X_frontal_SD_features=[]
for datta in tqdm_notebook(X_frontal_SD):
    features = average_bandpower_alpha(datta)
    X_frontal_SD_features.append(features)

X_frontal_SD_alpha=np.array(X_frontal_SD_features)
X_frontal_SD_alpha.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1656574856.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_frontal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [37]:
from tqdm import tqdm_notebook
X_frontal_SD_features=[]
for datta in tqdm_notebook(X_frontal_SD):
    features = average_bandpower_beta(datta)
    X_frontal_SD_features.append(features)

X_frontal_SD_beta=np.array(X_frontal_SD_features)
X_frontal_SD_beta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\613500282.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_frontal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [38]:
from tqdm import tqdm_notebook
X_frontal_SD_features=[]
for datta in tqdm_notebook(X_frontal_SD):
    features = average_bandpower_gamma(datta)
    X_frontal_SD_features.append(features)

X_frontal_SD_gamma=np.array(X_frontal_SD_features)
X_frontal_SD_gamma.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\3443139292.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_frontal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [39]:
from tqdm import tqdm_notebook
X_occipital_SD_features=[]
for datta in tqdm_notebook(X_occipital_SD):
    features = average_bandpower_delta(datta)
    X_occipital_SD_features.append(features)

X_occipital_SD_delta=np.array(X_occipital_SD_features)
X_occipital_SD_delta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1971129482.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_occipital_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [40]:
from tqdm import tqdm_notebook
X_occipital_SD_features=[]
for datta in tqdm_notebook(X_occipital_SD):
    features = average_bandpower_theta(datta)
    X_occipital_SD_features.append(features)

X_occipital_SD_theta=np.array(X_occipital_SD_features)
X_occipital_SD_theta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1435177507.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_occipital_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [41]:
from tqdm import tqdm_notebook
X_occipital_SD_features=[]
for datta in tqdm_notebook(X_occipital_SD):
    features = average_bandpower_alpha(datta)
    X_occipital_SD_features.append(features)

X_occipital_SD_alpha=np.array(X_occipital_SD_features)
X_occipital_SD_alpha.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2759367849.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_occipital_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [42]:
from tqdm import tqdm_notebook
X_occipital_SD_features=[]
for datta in tqdm_notebook(X_occipital_SD):
    features = average_bandpower_beta(datta)
    X_occipital_SD_features.append(features)

X_occipital_SD_beta=np.array(X_occipital_SD_features)
X_occipital_SD_beta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1000373613.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_occipital_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [43]:
from tqdm import tqdm_notebook
X_occipital_SD_features=[]
for datta in tqdm_notebook(X_occipital_SD):
    features = average_bandpower_gamma(datta)
    X_occipital_SD_features.append(features)

X_occipital_SD_gamma=np.array(X_occipital_SD_features)
X_occipital_SD_gamma.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\317465128.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_occipital_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [44]:
from tqdm import tqdm_notebook
X_parietal_SD_features=[]
for datta in tqdm_notebook(X_parietal_SD):
    features = average_bandpower_delta(datta)
    X_parietal_SD_features.append(features)

X_parietal_SD_delta=np.array(X_parietal_SD_features)
X_parietal_SD_delta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1734153515.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_parietal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [45]:
from tqdm import tqdm_notebook
X_parietal_SD_features=[]
for datta in tqdm_notebook(X_parietal_SD):
    features = average_bandpower_theta(datta)
    X_parietal_SD_features.append(features)

X_parietal_SD_theta=np.array(X_parietal_SD_features)
X_parietal_SD_theta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\300120872.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_parietal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [46]:
from tqdm import tqdm_notebook
X_parietal_SD_features=[]
for datta in tqdm_notebook(X_parietal_SD):
    features = average_bandpower_alpha(datta)
    X_parietal_SD_features.append(features)

X_parietal_SD_alpha=np.array(X_parietal_SD_features)
X_parietal_SD_alpha.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\4239888457.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_parietal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [47]:
from tqdm import tqdm_notebook
X_parietal_SD_features=[]
for datta in tqdm_notebook(X_parietal_SD):
    features = average_bandpower_beta(datta)
    X_parietal_SD_features.append(features)

X_parietal_SD_beta=np.array(X_parietal_SD_features)
X_parietal_SD_beta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1795316931.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_parietal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [48]:
from tqdm import tqdm_notebook
X_parietal_SD_features=[]
for datta in tqdm_notebook(X_parietal_SD):
    features = average_bandpower_gamma(datta)
    X_parietal_SD_features.append(features)

X_parietal_SD_gamma=np.array(X_parietal_SD_features)
X_parietal_SD_gamma.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2998718499.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_parietal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [49]:
from tqdm import tqdm_notebook
X_temporal_SD_features=[]
for datta in tqdm_notebook(X_temporal_SD):
    features = average_bandpower_delta(datta)
    X_temporal_SD_features.append(features)

X_temporal_SD_delta=np.array(X_temporal_SD_features)
X_temporal_SD_delta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1098594215.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_temporal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [50]:
from tqdm import tqdm_notebook
X_temporal_SD_features=[]
for datta in tqdm_notebook(X_temporal_SD):
    features = average_bandpower_theta(datta)
    X_temporal_SD_features.append(features)

X_temporal_SD_theta=np.array(X_temporal_SD_features)
X_temporal_SD_theta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2126798678.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_temporal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [51]:
from tqdm import tqdm_notebook
X_temporal_SD_features=[]
for datta in tqdm_notebook(X_temporal_SD):
    features = average_bandpower_alpha(datta)
    X_temporal_SD_features.append(features)

X_temporal_SD_alpha=np.array(X_temporal_SD_features)
X_temporal_SD_alpha.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2004636236.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_temporal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [52]:
from tqdm import tqdm_notebook
X_temporal_SD_features=[]
for datta in tqdm_notebook(X_temporal_SD):
    features = average_bandpower_beta(datta)
    X_temporal_SD_features.append(features)

X_temporal_SD_beta=np.array(X_temporal_SD_features)
X_temporal_SD_beta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\828727793.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_temporal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [53]:
from tqdm import tqdm_notebook
X_temporal_SD_features=[]
for datta in tqdm_notebook(X_temporal_SD):
    features = average_bandpower_gamma(datta)
    X_temporal_SD_features.append(features)

X_temporal_SD_gamma=np.array(X_temporal_SD_features)
X_temporal_SD_gamma.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2866951538.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_temporal_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [54]:
from tqdm import tqdm_notebook
X_central_SD_features=[]
for datta in tqdm_notebook(X_central_SD):
    features = average_bandpower_delta(datta)
    X_central_SD_features.append(features)

X_central_SD_delta=np.array(X_central_SD_features)
X_central_SD_delta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2205996521.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_central_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [55]:
from tqdm import tqdm_notebook
X_central_SD_features=[]
for datta in tqdm_notebook(X_central_SD):
    features = average_bandpower_theta(datta)
    X_central_SD_features.append(features)

X_central_SD_theta=np.array(X_central_SD_features)
X_central_SD_theta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1070835115.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_central_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [56]:
from tqdm import tqdm_notebook
X_central_SD_features=[]
for datta in tqdm_notebook(X_central_SD):
    features = average_bandpower_alpha(datta)
    X_central_SD_features.append(features)

X_central_SD_alpha=np.array(X_central_SD_features)
X_central_SD_alpha.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\2265726124.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_central_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [57]:
from tqdm import tqdm_notebook
X_central_SD_features=[]
for datta in tqdm_notebook(X_central_SD):
    features = average_bandpower_beta(datta)
    X_central_SD_features.append(features)

X_central_SD_beta=np.array(X_central_SD_features)
X_central_SD_beta.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\1118039832.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_central_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

In [58]:
from tqdm import tqdm_notebook
X_central_SD_features=[]
for datta in tqdm_notebook(X_central_SD):
    features = average_bandpower_gamma(datta)
    X_central_SD_features.append(features)

X_central_SD_gamma=np.array(X_central_SD_features)
X_central_SD_gamma.shape

C:\Users\suvan\AppData\Local\Temp\ipykernel_19132\69214963.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for datta in tqdm_notebook(X_central_SD):


  0%|          | 0/210 [00:00<?, ?it/s]

(210,)

## T-test

In [59]:
from scipy import stats

### Frontal

In [60]:
t_stat_frontal, p_frontal_t = stats.ttest_ind(X_frontal_SD_delta, X_frontal_NS_delta, equal_var=False)
print("FRONTAL: \n","T_stat:\n", t_stat_frontal,"\n","P_stat:\n", p_frontal_t)

FRONTAL: 
 T_stat:
 2.6451719441483323 
 P_stat:
 0.00846998838124606


In [61]:
t_stat_frontal, p_frontal_t = stats.ttest_ind(X_frontal_SD_theta, X_frontal_NS_theta, equal_var=False)
print("FRONTAL: \n","T_stat:\n", t_stat_frontal,"\n","P_stat:\n", p_frontal_t)

FRONTAL: 
 T_stat:
 3.4443389938312174 
 P_stat:
 0.0006498503484355084


In [62]:
t_stat_frontal, p_frontal_t = stats.ttest_ind(X_frontal_SD_alpha, X_frontal_NS_alpha, equal_var=False)
print("FRONTAL: \n","T_stat:\n", t_stat_frontal,"\n","P_stat:\n", p_frontal_t)

FRONTAL: 
 T_stat:
 -1.4925979910546145 
 P_stat:
 0.13662912830469776


In [63]:
t_stat_frontal, p_frontal_t = stats.ttest_ind(X_frontal_SD_beta, X_frontal_NS_beta, equal_var=False)
print("FRONTAL: \n","T_stat:\n", t_stat_frontal,"\n","P_stat:\n", p_frontal_t)

FRONTAL: 
 T_stat:
 3.115685133955737 
 P_stat:
 0.001978485396834843


In [64]:
t_stat_frontal, p_frontal_t = stats.ttest_ind(X_frontal_SD_gamma, X_frontal_NS_gamma, equal_var=False)
print("FRONTAL: \n","T_stat:\n", t_stat_frontal,"\n","P_stat:\n", p_frontal_t)

FRONTAL: 
 T_stat:
 2.404238300859921 
 P_stat:
 0.016646554996437295


### Occipital

In [65]:
t_stat_occipital, p_occipital_t = stats.ttest_ind(X_occipital_SD_delta, X_occipital_NS_delta, equal_var=False)
print("OCCIPITAL: \n","T_stat:\n", t_stat_occipital,"\n","P_stat:\n", p_occipital_t)

OCCIPITAL: 
 T_stat:
 3.224426360138549 
 P_stat:
 0.0013679126429930937


In [66]:
t_stat_occipital, p_occipital_t = stats.ttest_ind(X_occipital_SD_theta, X_occipital_NS_theta, equal_var=False)
print("OCCIPITAL: \n","T_stat:\n", t_stat_occipital,"\n","P_stat:\n", p_occipital_t)

OCCIPITAL: 
 T_stat:
 3.190635248499001 
 P_stat:
 0.0015269980309934204


In [67]:
t_stat_occipital, p_occipital_t = stats.ttest_ind(X_occipital_SD_alpha, X_occipital_NS_alpha, equal_var=False)
print("OCCIPITAL: \n","T_stat:\n", t_stat_occipital,"\n","P_stat:\n", p_occipital_t)

OCCIPITAL: 
 T_stat:
 -0.7206880678285107 
 P_stat:
 0.4716438506515731


In [68]:
t_stat_occipital, p_occipital_t = stats.ttest_ind(X_occipital_SD_beta, X_occipital_NS_beta, equal_var=False)
print("OCCIPITAL: \n","T_stat:\n", t_stat_occipital,"\n","P_stat:\n", p_occipital_t)

OCCIPITAL: 
 T_stat:
 1.2298719758423726 
 P_stat:
 0.21943681474596047


In [69]:
t_stat_occipital, p_occipital_t = stats.ttest_ind(X_occipital_SD_gamma, X_occipital_NS_gamma, equal_var=False)
print("OCCIPITAL: \n","T_stat:\n", t_stat_occipital,"\n","P_stat:\n", p_occipital_t)

OCCIPITAL: 
 T_stat:
 0.7393864710818056 
 P_stat:
 0.46008967332968576


#### Parietal

In [70]:
t_stat_parietal, p_parietal_t = stats.ttest_ind(X_parietal_SD_delta, X_parietal_NS_delta, equal_var=False)
print("PARIETAL: \n","T_stat:\n", t_stat_parietal,"\n","P_stat:\n", p_parietal_t)

PARIETAL: 
 T_stat:
 3.389283351705676 
 P_stat:
 0.0007729313870826447


In [71]:
t_stat_parietal, p_parietal_t  = stats.ttest_ind(X_parietal_SD_theta, X_parietal_NS_theta, equal_var=False)
print("PARIETAL: \n","T_stat:\n", t_stat_parietal,"\n","P_stat:\n", p_parietal_t)

PARIETAL: 
 T_stat:
 1.7712573318638782 
 P_stat:
 0.07723983063813303


In [72]:
t_stat_parietal, p_parietal_t  = stats.ttest_ind(X_parietal_SD_alpha, X_parietal_NS_alpha, equal_var=False)
print("PARIETAL: \n","T_stat:\n", t_stat_parietal,"\n","P_stat:\n", p_parietal_t)

PARIETAL: 
 T_stat:
 -1.9539102190075526 
 P_stat:
 0.05177165033474355


In [73]:
t_stat_parietal, p_parietal_t  = stats.ttest_ind(X_parietal_SD_beta, X_parietal_NS_beta, equal_var=False)
print("PARIETAL: \n","T_stat:\n", t_stat_parietal,"\n","P_stat:\n", p_parietal_t)

PARIETAL: 
 T_stat:
 0.7545623025933463 
 P_stat:
 0.4509557826779893


In [74]:
t_stat_parietal, p_parietal_t  = stats.ttest_ind(X_parietal_SD_gamma, X_parietal_NS_gamma, equal_var=False)
print("PARIETAL: \n","T_stat:\n", t_stat_parietal,"\n","P_stat:\n", p_parietal_t)

PARIETAL: 
 T_stat:
 0.5282458783678914 
 P_stat:
 0.5976707264925754


#### temporal

In [75]:
t_stat_temporal, p_temporal_t = stats.ttest_ind(X_temporal_SD_delta, X_temporal_NS_delta, equal_var=False)
print("TEMPORAL: \n","T_stat:\n", t_stat_temporal,"\n","P_stat:\n", p_temporal_t)

TEMPORAL: 
 T_stat:
 0.7800560901032393 
 P_stat:
 0.4357982741754731


In [76]:
t_stat_temporal, p_temporal_t = stats.ttest_ind(X_temporal_SD_theta, X_temporal_NS_theta, equal_var=False)
print("TEMPORAL: \n","T_stat:\n", t_stat_temporal,"\n","P_stat:\n", p_temporal_t)

TEMPORAL: 
 T_stat:
 3.211190630175643 
 P_stat:
 0.001438373250871589


In [77]:
t_stat_temporal, p_temporal_t = stats.ttest_ind(X_temporal_SD_alpha, X_temporal_NS_alpha, equal_var=False)
print("TEMPORAL: \n","T_stat:\n", t_stat_temporal,"\n","P_stat:\n", p_temporal_t)

TEMPORAL: 
 T_stat:
 -0.22167269855097244 
 P_stat:
 0.824676738213244


In [78]:
t_stat_temporal, p_temporal_t = stats.ttest_ind(X_temporal_SD_beta, X_temporal_NS_beta, equal_var=False)
print("TEMPORAL: \n","T_stat:\n", t_stat_temporal,"\n","P_stat:\n", p_temporal_t)

TEMPORAL: 
 T_stat:
 0.18460963186599924 
 P_stat:
 0.8536413279123116


In [79]:
t_stat_temporal, p_temporal_t = stats.ttest_ind(X_temporal_SD_gamma, X_temporal_NS_gamma, equal_var=False)
print("TEMPORAL: \n","T_stat:\n", t_stat_temporal,"\n","P_stat:\n", p_temporal_t)

TEMPORAL: 
 T_stat:
 -0.1169535676699137 
 P_stat:
 0.9069539130155296


#### Central

In [80]:
t_stat_central, p_central_t = stats.ttest_ind(X_central_SD_delta, X_central_NS_delta, equal_var=False)
print("CENTRAL: \n","T_stat:\n", t_stat_central,"\n","P_stat:\n", p_central_t)

CENTRAL: 
 T_stat:
 2.3887606238515 
 P_stat:
 0.01739893422691499


In [81]:
t_stat_central, p_central_t = stats.ttest_ind(X_central_SD_theta, X_central_NS_theta, equal_var=False)
print("CENTRAL: \n","T_stat:\n", t_stat_central,"\n","P_stat:\n", p_central_t)

CENTRAL: 
 T_stat:
 2.3254758896422856 
 P_stat:
 0.020524525016698162


In [82]:
t_stat_central, p_central_t = stats.ttest_ind(X_central_SD_alpha, X_central_NS_alpha, equal_var=False)
print("CENTRAL: \n","T_stat:\n", t_stat_central,"\n","P_stat:\n", p_central_t)

CENTRAL: 
 T_stat:
 -2.8204006290296415 
 P_stat:
 0.0050976963991853025


In [83]:
t_stat_central, p_central_t = stats.ttest_ind(X_central_SD_beta, X_central_NS_beta, equal_var=False)
print("CENTRAL: \n","T_stat:\n", t_stat_central,"\n","P_stat:\n", p_central_t)

CENTRAL: 
 T_stat:
 0.594087392190851 
 P_stat:
 0.5527777596011879


In [84]:
t_stat_central, p_central_t = stats.ttest_ind(X_central_SD_gamma, X_central_NS_gamma, equal_var=False)
print("CENTRAL: \n","T_stat:\n", t_stat_central,"\n","P_stat:\n", p_central_t)

CENTRAL: 
 T_stat:
 -0.20730551526993998 
 P_stat:
 0.8358778643282363
